In [ ]:
import pandas as pd
from bs4 import BeautifulSoup as bs
from tqdm import tqdm
import regex as re
import os
import json

from rdflib import Graph, Namespace, URIRef, Literal, BNode
from rdflib.namespace import RDF, RDFS


In [ ]:
# init operations
BASE_URI = 'http://bibframe.example.org/'

WORK = Namespace(BASE_URI + "works/")
INSTANCE = Namespace(BASE_URI + "instances/")
ITEM = Namespace(BASE_URI + "items/")
AGENT = Namespace(BASE_URI + "agents/")
SUBJECT = Namespace(BASE_URI + "subjects/")

BF = Namespace('http://id.loc.gov/ontologies/bibframe/')


In [35]:
with open('entities/entities_dict.json', 'r', encoding='utf-8') as jfile:
    entities_dict = json.load(jfile)

with open('entities/flow_control.json', 'r', encoding='utf-8') as jfile:
    flow_control = json.load(jfile)

with open('archived_urls.json', 'r', encoding='utf-8') as jfile:
    archived_urls = json.load(jfile)

lcsh_mapping = pd.read_excel('new_pbl_to_lcsh_mapping.xlsx')
lcsh_mapping = {row['new_pbl']:row['lcsh'] for _, row in lcsh_mapping.iterrows() if str(row['lcsh']) != 'nan'}

bn_topics = pd.read_excel('lcsh_to_bn.xlsx')
bn_topics = {row['lcsh']:{'bn_url':row['bn_url'], 'bn_label':row['bn_label']} for _, row in bn_topics.iterrows()}

In [36]:
def manual_df_to_dict(path):
    df = pd.read_excel(path, sheet_name='Posts').fillna('')
    df = df[df['do PBL'] == True]
    return df.to_dict(orient='records')

def clear_viaf_uri(url):
    if uri := re.match(r'https://viaf\.org/viaf/\d+', url):
        return uri.group(0) + '/'
    
def clean_records_dct(records):
    records_deduplicated = []
    records_dct_to_clean = {}
    for elem in records:
        link = elem.get('Link')
        if not link:
            continue
        if 'krystynajanda' in link:
            records_deduplicated.append(elem)
        else:
            if link not in records_dct_to_clean:
                records_dct_to_clean[link] = elem
    
    for key, value in records_dct_to_clean.items():
        records_deduplicated.append(value)
    
    return records_deduplicated


In [39]:
def add_record_to_graph(graph, record_dict):
    g = graph
    work_id = str(flow_control.get('work_last_id') + 1).zfill(8)
    work = WORK[work_id]
    flow_control['work_last_id'] += 1
    g.add((work, RDF.type, BF.Work))
    
    instance_id = str(flow_control.get('instance_last_id') + 1).zfill(8)
    instance = INSTANCE[instance_id]
    flow_control['instance_last_id'] += 1
    g.add((work, BF.hasInstance, instance))
    g.add((instance, RDF.type, BF.Instance))
    g.add((instance, BF.instanceOf, work))
    
    item_id = str(flow_control.get('item_last_id') + 1).zfill(8)
    item = ITEM[item_id]
    flow_control['item_last_id'] += 1
    g.add((instance, BF.hasItem, item))
    g.add((item, RDF.type, BF.Item))
    g.add((item, BF.itemOf, instance))

    for key, value in record_dict.items():
        value = str(value)
        if not value:
            continue
        
        match key:
            case 'Link':
                g.add((instance, BF.electronicLocator, URIRef(value.strip())))
                g.add((item, BF.electronicLocator, URIRef(value.strip())))
                
            case 'Data publikacji':
                g.add((instance, BF.originDate, Literal(value.strip())))
                
            case 'Autor':
                for author in value.split('|'):
                    author_dct = entities_dict['authors'].get(author.strip())
                    if author_dct:
                        author_id, viaf_uri, viaf_label = author_dct['author_id'], author_dct['viaf_uri'], author_dct['viaf_label'] # it is possible to use locals().update(author_dct)
                        label = viaf_label if viaf_label else author
                        
                        # create an Agent
                        agent = AGENT[author_id]
                        g.add((agent, RDF.type, BF.Agent))
                        g.add((agent, RDF.type, BF.Person))
                        g.add((agent, RDFS.label, Literal(label)))
                        if viaf_uri and (agent, BF.identifiedBy, None) not in g:
                            identifier = BNode()
                            g.add((identifier, RDF.type, BF.Identifier))
                            g.add((identifier, RDF.value, Literal(viaf_uri)))
                            g.add((agent, BF.identifiedBy, identifier))
        
                        # add Agent
                        contribution = BNode()
                        g.add((work, BF.contribution, contribution))
                        g.add((contribution, RDF.type, BF.Contribution))
                        g.add((contribution, RDF.type, BF.PrimaryContribution))
                        g.add((contribution, BF.agent, agent))

                        # add role
                        author_role = URIRef('http://id.loc.gov/vocabulary/relators/aut')
                        g.add((contribution, BF.role, author_role))
                        g.add((author_role, RDF.type, BF.Role))
                        g.add((author_role, RDFS.label, Literal('author')))
                
            case 'do PBL':
                pass
            
            case 'VIAF autor 1':
                pass
            
            case 'VIAF autor 2':
                pass
            
            case 'VIAF autor 3':
                pass
            
            case 'Sekcja':
                topic_dct = entities_dict['subjects'].get(value.strip())
                if topic_dct:
                    topic_id = topic_dct['topic_id']
                    topic = SUBJECT[topic_id]
                    g.add((work, BF.subject, topic))
                    g.add((topic, RDF.type, BF.Topic))
                    g.add((topic, RDFS.label, Literal(value.strip())))
                
            case 'Tytuł artykułu':                
                title = BNode()
                g.add((work, BF.title, title))
                g.add((title, RDF.type, BF.Title))
                g.add((title, BF.mainTitle, Literal(value.strip())))
                
            case 'Opis':
                summary = BNode()
                g.add((work, BF.summary, summary))
                g.add((summary, RDF.type, BF.Summary))
                g.add((summary, RDFS.label, Literal(value.strip())))
                
            case 'Numer':
                enumeration = BNode()
                g.add((item, BF.enumerationAndChronology, enumeration))
                g.add((enumeration, RDF.type, BF.Enumeration))
                if isinstance(value, float): value = int(value)
                g.add((enumeration, RDFS.label, Literal(str(value).strip())))
            
            case 'Tagi':
                for tag in value.split('|'):
                    topic_dct = entities_dict['subjects'].get(tag.strip())
                    if topic_dct:
                        topic_id = topic_dct['topic_id']
                        topic = SUBJECT[topic_id]
                        g.add((work, BF.subject, topic))
                        g.add((topic, RDF.type, BF.Topic))
                        g.add((topic, RDFS.label, Literal(tag.strip())))

            case 'forma/gatunek':
                try:
                    genreform_uri = entities_dict['genreforms'].get(value)
                    genreform = URIRef(genreform_uri)
                except:
                    continue
                g.add((work, BF.genreForm, genreform))
                g.add((genreform, RDF.type, BF.GenreForm))
                g.add((genreform, RDFS.label, Literal(value.strip())))
                
            case 'hasła przedmiotowe':
                headings_splitted = [e.strip() for e in value.split('|')]
                for heading in headings_splitted:
                    lcsh_links = lcsh_mapping.get(heading)
                    if lcsh_links:
                        lcsh_links_splitted = lcsh_links.split(' | ')
                        for lcsh_url in lcsh_links_splitted:
                            subject_lcsh = URIRef(lcsh_url)
                            g.add((work, BF.subject, subject_lcsh))
                            g.add((subject_lcsh, RDF.type, BF.Topic))

                            bn_data = bn_topics.get(lcsh_url) 
                            if bn_data:
                                bn_url = bn_data.get('bn_url')
                                bn_label = bn_data.get('bn_label')
                                subject_bn = URIRef(bn_url)
                                g.add((work, BF.subject, subject_bn))
                                g.add((subject_bn, RDF.type, BF.Topic))
                                g.add((subject_bn, RDFS.label, Literal(bn_label.strip())))
            
            case 'zewnętrzny identyfikator bytu 1' | 'zewnętrzny identyfikator bytu 2' | 'zewnętrzny identyfikator bytu 3':
                if 'viaf.org' in value:
                    value = clear_viaf_uri(value)
                topic_dct = entities_dict['subjects'].get(value)
                if topic_dct:
                    match topic_dct['bibframe_type']:
                        case 'bf:Work':
                            uri = topic_dct['external_uri']
                            subject_work = URIRef(uri)
                            g.add((work, BF.subject, subject_work))
                            g.add((subject_work, RDF.type, BF.Work))
                            title = BNode()
                            g.add((subject_work, BF.title, title))
                            g.add((title, RDF.type, BF.Title))
                            g.add((title, BF.mainTitle, Literal(topic_dct['label'])))
                        case 'bf:MovingImage':
                            uri = topic_dct['external_uri']
                            movie_work = URIRef(uri)
                            g.add((work, BF.subject, movie_work))
                            g.add((movie_work, RDF.type, BF.Work))
                            g.add((movie_work, RDF.type, BF.MovingImage))
                            title = BNode()
                            g.add((movie_work, BF.title, title))
                            g.add((title, RDF.type, BF.Title))
                            g.add((title, BF.mainTitle, Literal(topic_dct['label'])))
                        case 'bf:Organization':
                            uri = topic_dct['external_uri']
                            subject_organization = URIRef(uri)
                            g.add((work, BF.subject, subject_organization))
                            g.add((subject_organization, RDF.type, BF.Organization))
                            g.add((subject_organization, RDFS.label, Literal(topic_dct['label'])))
                        case 'bf:Person':
                            uri = topic_dct['external_uri']
                            subject_person = URIRef(uri)
                            g.add((work, BF.subject, subject_person))
                            g.add((subject_person, RDF.type, BF.Person))
                            g.add((subject_person, RDFS.label, Literal(topic_dct['label'])))
                        case 'bf:Event':
                            uri = topic_dct['external_uri']
                            subject_event = URIRef(uri)
                            g.add((work, BF.subject, subject_event))
                            g.add((subject_event, RDF.type, BF.Event))
                            g.add((subject_event, RDFS.label, Literal(topic_dct['label'])))
                        case 'bf:Place':
                            uri = topic_dct['external_uri']
                            subject_place = URIRef(uri)
                            g.add((work, BF.subject, subject_place))
                            g.add((subject_place, RDF.type, BF.Place))
                            g.add((subject_place, RDFS.label, Literal(topic_dct['label'])))
            case 'byt 1':
                pass

            case 'byt 2':
                pass
            
            case 'byt 3':
                pass
            
            case 'adnotacje':
                pass
            
            case 'Linki zewnętrzne':
                pass
            
            case 'Linki do zdjęć':
                pass

            case _:
                pass
    
    # internet archive url
    rec_url = record_dict.get('Link')
    if (internet_archive_url := archived_urls.get(rec_url)):
        g.add((instance, BF.electronicLocator, URIRef(internet_archive_url.strip())))
        ia_item_id = str(flow_control.get('item_last_id') + 1).zfill(8)
        ia_item = ITEM[ia_item_id]
        flow_control['item_last_id'] += 1
        g.add((instance, BF.hasItem, ia_item))
        g.add((ia_item, RDF.type, BF.Item))
        g.add((ia_item, BF.itemOf, instance))
        g.add((ia_item, BF.electronicLocator, URIRef(internet_archive_url.strip())))


def save_graph(graph, name='output', dst='bibframe_output/'):
    graph.serialize(destination=f"{dst}{name}.bibframe.xml", format="pretty-xml")


In [ ]:
processed_records_counter = 0
list_of_files = os.listdir('to_bibframe/automatic_filtered/')
for file in list_of_files:
    print(file)
    filename = file.split('.')[0]
    path = 'to_bibframe/automatic_filtered/' + file
    # records = manual_df_to_dict(path)
    with open(path, 'r', encoding='utf-8') as json_file:
        records = json.load(json_file)
    records = clean_records_dct(records)
    g = Graph()
    g.bind("bf", BF)
    g.bind("work", WORK)
    g.bind("instance", INSTANCE)
    g.bind("item", ITEM)
    g.bind("agent", AGENT)
    g.bind("subject", SUBJECT)
    for rec in tqdm(records, desc='Adding records to graph...'):
        add_record_to_graph(g, rec)
        processed_records_counter += 1
    save_graph(g, name=filename, dst='bibframe_output/automatic/')

print('PROCESSED RECORDS:', processed_records_counter)